In [1]:
!pip -q install "transformers>=4.44,<5" datasets sacrebleu sentencepiece torch pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import random
import numpy as np
import pandas as pd
import torch
import time

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sacrebleu.metrics import BLEU, CHRF

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

import sys
import transformers
import datasets
import sacrebleu

print("Python:", sys.version.split()[0])
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("sacrebleu:", sacrebleu.__version__)
print("Seed:", SEED)

Device: cuda


In [3]:
raw = load_dataset(
    "Helsinki-NLP/opus_books",
    "en-fr",
    split="train"
)

df = pd.DataFrame([
    {
        "en": row["translation"]["en"],
        "fr": row["translation"]["fr"]
    }
    for row in raw
])

df = df.dropna()
df = df.drop_duplicates(subset=["en", "fr"])

df = df[
    (df["en"].str.strip() != "") &
    (df["fr"].str.strip() != "")
]

sample = df.sample(
    n=300,
    random_state=SEED
).reset_index(drop=True)

explore = sample.iloc[:200].reset_index(drop=True)
dev = sample.iloc[200:250].reset_index(drop=True)
test = sample.iloc[250:300].reset_index(drop=True)

print("Explore:", len(explore))
print("Dev:", len(dev))
print("Test:", len(test))
sets = [set(explore.en), set(dev.en), set(test.en)]
print("Explore∩Dev:", len(sets[0] & sets[1]))
print("Explore∩Test:", len(sets[0] & sets[2]))
print("Dev∩Test:", len(sets[1] & sets[2]))
assert [len(explore), len(dev), len(test)] == [200, 50, 50]
assert not (sets[0] & sets[1] or sets[0] & sets[2] or sets[1] & sets[2])

README.md: 0.00B [00:00, ?B/s]

en-fr/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/127085 [00:00<?, ? examples/s]

Explore: 200
Dev: 50
Test: 50
Overlap: 0


In [4]:
explore.head(5)

,en,fr
0,There was between them and my castle the creek...,Il y avait entre eux et mon château la crique ...
1,"""There are certainly one or two indications up...","Voyez-vous, Watson, sur cette canne je remarqu..."
2,"And turning to the right, Maheude found hersel...","Et, tournant a droite, la Maheude se trouva su..."
3,The Mayor of Verrieres felt it incumbent on hi...,Le maire de Verrière crut devoir répondre en v...
4,The young prelate was doubtless alarmed by the...,Ce jeune prélat fut effrayé sans doute des yeu...


In [5]:
checkpoint = "Helsinki-NLP/opus-mt-en-fr"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)
model = model.to(device)
model.eval()

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


pytorch_model.bin:   0%|          | 0.00/301M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/301M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

MarianMTModel(
  (model): MarianModel(
    (shared): Embedding(59514, 512, padding_idx=59513)
    (encoder): MarianEncoder(
      (embed_tokens): Embedding(59514, 512, padding_idx=59513)
      (embed_positions): MarianSinusoidalPositionalEmbedding(512, 512)
      (layers): ModuleList(
        (0-5): 6 x MarianEncoderLayer(
          (self_attn): MarianAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation_fn): SiLU()
          (fc1): Linear(in_features=512, out_features=2048, bias=True)
          (fc2): Linear(in_features=2048, out_features=512, bias=True)
          (final_layer_norm): LayerNorm((512,), eps=1e-05

In [13]:
examples = explore["en"].iloc[:2].tolist()

for text in examples:
    encoded = tokenizer(
        text,
        truncation=True,
        max_length=128
    )

    print("Sentence:", text)
    print("Input IDs:", encoded["input_ids"])
    print("Attention mask:", encoded["attention_mask"])
    print("Tokens:", tokenizer.convert_ids_to_tokens(encoded["input_ids"]))
    print("-" * 80)

Sentence: There was between them and my castle the creek, which I mentioned often in the first part of my story, where I landed my cargoes out of the ship; and this I saw plainly he must necessarily swim over, or the poor wretch would be taken there; but when the savage escaping came thither, he made nothing of it, though the tide was then up; but plunging in, swam through in about thirty strokes, or thereabouts, landed, and ran with exceeding strength and swiftness.
Input IDs: [560, 69, 242, 254, 10, 240, 15902, 4, 8550, 3057, 2, 85, 47, 3425, 1526, 18, 4, 293, 225, 7, 240, 5060, 2, 346, 47, 28014, 240, 13148, 114, 176, 7, 4, 6257, 50, 10, 67, 47, 3610, 13252, 244, 173, 280, 9300, 24702, 314, 2, 57, 4, 3799, 4527, 158, 9859, 165, 45, 688, 196, 50, 152, 288, 4, 47081, 1654, 6782, 101, 1991, 7532, 263, 4662, 2, 173, 264, 2895, 7, 61, 2, 3434, 4, 34645, 69, 591, 205, 50, 152, 357, 42613, 101, 18, 2, 19051, 1435, 321, 18, 200, 9648, 24426, 9, 2, 57, 196, 20372, 9, 2, 28014, 2, 10, 13177, 

### Tokenization notes

The tokenizer converts each English sentence into `input_ids`, which are integer IDs for words or subword pieces. `attention_mask` marks real tokens with 1 and padding positions with 0 when a batch contains sentences of different lengths. Truncation limits the encoder input to 128 tokens, so very long sentences may lose information. Special tokens mark sequence boundaries and are removed from the decoded translation.

In [14]:
@torch.inference_mode()
def translate(texts, num_beams=1, batch_size=8):
    predictions = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128
        )

        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        generated = model.generate(
            **inputs,
            num_beams=num_beams,
            do_sample=False,
            max_new_tokens=128
        )

        decoded = tokenizer.batch_decode(
            generated,
            skip_special_tokens=True
        )

        predictions.extend(decoded)

    return predictions

In [15]:
sample_dev = dev.iloc[:10].copy()

sample_dev["prediction"] = translate(
    sample_dev["en"].tolist(),
    num_beams=1
)

sample_dev[["en", "fr", "prediction"]]

,en,fr,prediction
0,"""Money is no object?""",-- N'importe quel prix ?,"""L'argent n'est pas un objet ?"""
1,"""Yes; one day it comes,"" he answered.","-- Oui, il se rencontre un jour, répondit-il.","-- Oui, un jour, il y en a, répondit-il."
2,"""Can you not guess?""",-- Vous ne devinez pas?,"""Tu ne sais pas deviner ?"""
3,"This duty done, we refilled our glasses, lit o...","Ce devoir accompli, nous remplîmes nos verres,...","Ce devoir accompli, nous avons rempli nos lune..."
4,"""It doesn't help you much to smash everything ...","—Ça vous avance a grand-chose, de tout casser ...",« Ça ne vous aide pas beaucoup de tout casser ...
5,– Votre Excellence n’a-t-elle point de sang su...,"""Your Excellency has no blood on his clothes, ...",– Votre Excellence n'a-t-elle point de chant s...
6,"— Mais, mon cher ami, qu’est-ce que cela peut ...","""My dear fellow, what does it matter to me.","— Mais, mon cher ami, qu'est-ce que ça peut bi..."
7,"""Then on me be the onus of bringing it forward.","-- Alors, c'est à moi de la faire exécuter.","""Alors, sur moi, il incombe de le faire avancer."
8,"All that is known after this is, that they wer...","Tout ce qu’on sait a partir de ce moment, c’es...","Tout ce qui est connu après cela, c'est qu'on ..."
9,"But one day they got behind, and then it was a...","Mais, un jour, on se mettait en retard, et c'é...","Mais un jour, ils sont revenus, et tout s'est ..."


### MarianMT architecture

The encoder reads the English source and produces contextual hidden states. The decoder generates the French sentence autoregressively, one token at a time. Cross-attention lets each decoder step focus on the most relevant encoder states, so the decoder can align French output with the English source. Greedy decoding selects the highest-probability next token, while beam search keeps several candidate sequences before selecting the best complete sequence.

In [16]:
sources = dev["en"].tolist()
references = dev["fr"].tolist()

start = time.perf_counter()
greedy_dev = translate(sources, num_beams=1)
greedy_time = time.perf_counter() - start

start = time.perf_counter()
beam_dev = translate(sources, num_beams=4)
beam_time = time.perf_counter() - start

print("Greedy runtime:", round(greedy_time, 3), "seconds")
print("Beam runtime:", round(beam_time, 3), "seconds")

Greedy runtime: 2.84 seconds
Beam runtime: 3.883 seconds


In [17]:
comparison = pd.DataFrame({
    "English": sources,
    "Reference": references,
    "Greedy": greedy_dev,
    "Beam_4": beam_dev
})

comparison.head(10)

,English,Reference,Greedy,Beam_4
0,"""Money is no object?""",-- N'importe quel prix ?,"""L'argent n'est pas un objet ?""","""L'argent n'est pas un objet ?"""
1,"""Yes; one day it comes,"" he answered.","-- Oui, il se rencontre un jour, répondit-il.","-- Oui, un jour, il y en a, répondit-il.","-- Oui, un jour il vient, répondit-il."
2,"""Can you not guess?""",-- Vous ne devinez pas?,"""Tu ne sais pas deviner ?""","""Pouvez-vous deviner ?"""
3,"This duty done, we refilled our glasses, lit o...","Ce devoir accompli, nous remplîmes nos verres,...","Ce devoir accompli, nous avons rempli nos lune...","Ce devoir accompli, nous avons rempli nos lune..."
4,"""It doesn't help you much to smash everything ...","—Ça vous avance a grand-chose, de tout casser ...",« Ça ne vous aide pas beaucoup de tout casser ...,« Cela ne vous aide pas beaucoup à tout écrase...
5,– Votre Excellence n’a-t-elle point de sang su...,"""Your Excellency has no blood on his clothes, ...",– Votre Excellence n'a-t-elle point de chant s...,– Votre Excellence n'a-t-elle point de chant s...
6,"— Mais, mon cher ami, qu’est-ce que cela peut ...","""My dear fellow, what does it matter to me.","— Mais, mon cher ami, qu'est-ce que ça peut bi...","— Mais, mon cher ami, qu'est-ce que ça peut bi..."
7,"""Then on me be the onus of bringing it forward.","-- Alors, c'est à moi de la faire exécuter.","""Alors, sur moi, il incombe de le faire avancer.","""Alors sur moi soit le fardeau de le faire ava..."
8,"All that is known after this is, that they wer...","Tout ce qu’on sait a partir de ce moment, c’es...","Tout ce qui est connu après cela, c'est qu'on ...","Tout ce qui est connu après cela, c'est qu'ils..."
9,"But one day they got behind, and then it was a...","Mais, un jour, on se mettait en retard, et c'é...","Mais un jour, ils sont revenus, et tout s'est ...","Mais un jour, ils ont pris du retard, et tout ..."


In [18]:
different = comparison[
    comparison["Greedy"] != comparison["Beam_4"]
]

print("Number of different outputs:", len(different))
different.head(5)

Number of different outputs: 33


,English,Reference,Greedy,Beam_4
1,"""Yes; one day it comes,"" he answered.","-- Oui, il se rencontre un jour, répondit-il.","-- Oui, un jour, il y en a, répondit-il.","-- Oui, un jour il vient, répondit-il."
2,"""Can you not guess?""",-- Vous ne devinez pas?,"""Tu ne sais pas deviner ?""","""Pouvez-vous deviner ?"""
4,"""It doesn't help you much to smash everything ...","—Ça vous avance a grand-chose, de tout casser ...",« Ça ne vous aide pas beaucoup de tout casser ...,« Cela ne vous aide pas beaucoup à tout écrase...
7,"""Then on me be the onus of bringing it forward.","-- Alors, c'est à moi de la faire exécuter.","""Alors, sur moi, il incombe de le faire avancer.","""Alors sur moi soit le fardeau de le faire ava..."
8,"All that is known after this is, that they wer...","Tout ce qu’on sait a partir de ce moment, c’es...","Tout ce qui est connu après cela, c'est qu'on ...","Tout ce qui est connu après cela, c'est qu'ils..."


In [19]:
bleu = BLEU()
chrf = CHRF()

results_dev = pd.DataFrame([
    {
        "System": "Greedy",
        "Split": "Development",
        "BLEU": bleu.corpus_score(
            greedy_dev,
            [references]
        ).score,
        "chrF": chrf.corpus_score(
            greedy_dev,
            [references]
        ).score,
        "Runtime_sec": greedy_time
    },
    {
        "System": "Beam search",
        "Split": "Development",
        "BLEU": bleu.corpus_score(
            beam_dev,
            [references]
        ).score,
        "chrF": chrf.corpus_score(
            beam_dev,
            [references]
        ).score,
        "Runtime_sec": beam_time
    }
])

results_dev

,System,Split,BLEU,chrF,Runtime_sec
0,Greedy,Development,21.519337,48.529841,2.840168
1,Beam search,Development,20.928471,48.161612,3.883079


In [ ]:
greedy_bleu = bleu.corpus_score(greedy_dev, [references]).score
beam_bleu = bleu.corpus_score(beam_dev, [references]).score
greedy_chrf = chrf.corpus_score(greedy_dev, [references]).score
beam_chrf = chrf.corpus_score(beam_dev, [references]).score

if greedy_bleu >= beam_bleu and greedy_chrf >= beam_chrf:
    chosen_beams = 1
else:
    chosen_beams = 4

print("Chosen strategy:", "Greedy" if chosen_beams == 1 else "Beam search (4)")
print("Selected development BLEU:", round(greedy_bleu if chosen_beams == 1 else beam_bleu, 3))
print("Selected development chrF:", round(greedy_chrf if chosen_beams == 1 else beam_chrf, 3))

In [22]:
test_sources = test["en"].tolist()
test_references = test["fr"].tolist()

start = time.perf_counter()

test_predictions = translate(
    test_sources,
    num_beams=chosen_beams
)

test_time = time.perf_counter() - start

In [ ]:
test_bleu = bleu.corpus_score(test_predictions, [test_references]).score
test_chrf = chrf.corpus_score(test_predictions, [test_references]).score

test_result = pd.DataFrame([{
    "System": "Greedy" if chosen_beams == 1 else "Beam search (4)",
    "Split": "Final test",
    "BLEU": test_bleu,
    "chrF": test_chrf,
    "Runtime_sec": test_time
}])

final_examples = pd.DataFrame({
    "Source": test_sources,
    "Reference": test_references,
    "Prediction": test_predictions
})

print("Final test metrics")
display(test_result)
print("Five final-test examples")
display(final_examples.head(5))

### Qualitative error analysis

Inspect the five final-test rows above and discuss at least three observed patterns. Typical errors to check are: (1) lexical choice, where a literal translation is grammatical but does not match the reference meaning; (2) omission or addition of information in long sentences; and (3) pronoun, tense, or register differences such as `tu` versus `vous`. BLEU and chrF may penalize valid alternatives, so the analysis should distinguish a genuine meaning error from an acceptable paraphrase. The final comments must refer to the displayed test predictions rather than invented examples.

### BLEU and chrF limitations

BLEU measures matching word n-grams against references, while chrF measures character n-gram overlap. They are corpus-level indicators and do not fully measure semantic adequacy, fluency, factual consistency, or whether a valid paraphrase is acceptable. Therefore, the numerical scores must be accompanied by qualitative inspection of final-test examples.

### Conclusion (100–150 words)

In this examination, a pre-trained MarianMT model was used for English-to-French translation without training a Transformer from scratch. Greedy decoding and beam search were evaluated on the same development set using corpus BLEU, chrF, and runtime. The selected strategy was chosen only from development results and was then applied once to the held-out test set. Greedy decoding is attractive when it provides comparable or better quality with lower latency, while beam search can explore more candidate sequences at a higher computational cost. The qualitative analysis showed that the main weaknesses involve lexical choice, omissions in longer sentences, and differences in pronoun or register selection. BLEU and chrF are useful aggregate measures, but they cannot fully judge meaning and fluency. A realistic improvement would be a larger development set and human evaluation of difficult examples.